# 00 · PyBaMM smoke test (LFP DFN)

Sysblade Battery Digital Twin — environment + physics-model sanity check.

**Goal.** Confirm the W1 environment can:
1. Import PyBaMM + PyTorch + supporting libs.
2. Run a Doyle-Fuller-Newman (DFN) simulation of an LFP cell using the `Prada2013` parameter set.
3. Produce voltage / SOC / temperature curves that match LFP's characteristic flat plateau.
4. Demonstrate a dynamic load profile (proxy for AI-rack transient) so we know the simulator can ingest non-constant currents — the prerequisite for modelling the LIC's role in W3.

If every cell here runs without error, **W1 environment is green** and we move to W2 (LSTM training on Severson).

In [ ]:
import sys, platform
import pybamm, torch, numpy as np, scipy, pandas as pd, matplotlib
import matplotlib.pyplot as plt

print(f'Python   : {sys.version.split()[0]}  ({platform.platform()})')
print(f'PyBaMM   : {pybamm.__version__}')
print(f'PyTorch  : {torch.__version__}  (CUDA={torch.cuda.is_available()})')
print(f'NumPy    : {np.__version__}')
print(f'SciPy    : {scipy.__version__}')
print(f'Pandas   : {pd.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')

## 1. Parameter set selection — LFP

Per v2.2 proposal §E.1 Tier-B, the chemistry is LFP (15S, 48 V nominal). PyBaMM ships several LFP parameter sets; we use **Prada 2013** (the canonical academic LFP DFN baseline). `Chen2020` is NMC and is shown only for comparison.

In [ ]:
params = pybamm.ParameterValues('Prada2013')
print(f"Nominal capacity      : {params['Nominal cell capacity [A.h]']} A·h")
print(f"Lower voltage cut-off : {params['Lower voltage cut-off [V]']} V")
print(f"Upper voltage cut-off : {params['Upper voltage cut-off [V]']} V")
print(f"Initial concentration : {params['Initial concentration in negative electrode [mol.m-3]']:.0f} mol/m^3 (anode)")
print(f"Ambient temperature   : {params['Ambient temperature [K]']} K")

## 2. Constant-current discharge (1C) — verify LFP plateau

LFP cells have a flat plateau around 3.2–3.3 V across the bulk of SOC; the voltage drops sharply only near 0% SOC. This is the chemistry signature we need to see.

In [ ]:
model = pybamm.lithium_ion.DFN()
experiment = pybamm.Experiment([
    'Discharge at 1C until 2.5 V',
    'Rest for 10 minutes',
])
sim = pybamm.Simulation(model, parameter_values=params, experiment=experiment)
sol = sim.solve()

t = sol['Time [s]'].entries
V = sol['Voltage [V]'].entries
I = sol['Current [A]'].entries
soc = sol['Discharge capacity [A.h]'].entries
soc_pct = 100 * (1 - soc / params['Nominal cell capacity [A.h]'])

print(f"Steps: {len(t)},  end time {t[-1]/60:.1f} min,  V end {V[-1]:.3f} V")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t/60, V, lw=1.6)
axes[0].set_xlabel('Time (min)'); axes[0].set_ylabel('Voltage (V)')
axes[0].set_title('LFP DFN · 1C discharge'); axes[0].grid(alpha=0.3)
axes[1].plot(soc_pct, V, lw=1.6, color='C3')
axes[1].set_xlabel('SOC (%)'); axes[1].set_ylabel('Voltage (V)')
axes[1].set_title('Voltage vs SOC (LFP plateau)'); axes[1].grid(alpha=0.3)
axes[1].invert_xaxis()
plt.tight_layout(); plt.show()

**Expected.** A plateau between roughly 3.2 V and 3.35 V across most of the discharge, then a steep drop near the end. If the curve looks linear from start to end, something is wrong with the parameter set.

## 3. Dynamic current profile — proxy for AI-rack transient

Per v2.2 §B.1 (2), GB200 racks exhibit ±30 % power swing in the 1–50 ms window. Here we feed the DFN a square-wave current alternating between 1C and 3C every 100 ms for 60 s. The point of this cell is **not** the resulting waveform itself (a single LFP cell can't actually plateau against millisecond pulses — that's exactly why we add the LIC in W3) but to confirm the simulator accepts arbitrary current functions, which is the API surface we'll need.

In [ ]:
# Pre-tabulate the pulsed profile, then feed PyBaMM an Interpolant.
# (Callable current functions receive a SYMBOLIC time, not a NumPy array,
# so np.floor / np.where don't work directly — we build the array first.)
dt = 0.01  # 10 ms resolution
t_grid = np.arange(0, 60 + dt, dt)
base = params['Nominal cell capacity [A.h]']  # 1C in A
i_grid = base * (1.0 + 2.0 * ((t_grid // 0.1).astype(int) % 2))

params_pulse = params.copy()
params_pulse['Current function [A]'] = pybamm.Interpolant(
    t_grid, i_grid, pybamm.t, name='pulsed_current', interpolator='linear'
)
sim_pulse = pybamm.Simulation(model, parameter_values=params_pulse)
sol_pulse = sim_pulse.solve([0, 60])

tp = sol_pulse['Time [s]'].entries
Vp = sol_pulse['Voltage [V]'].entries
Ip = sol_pulse['Current [A]'].entries

fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(tp, Ip, lw=0.9, color='C2'); ax[0].set_ylabel('I (A)'); ax[0].grid(alpha=0.3)
ax[0].set_title('Pulsed current profile (1C / 3C, 100 ms square)')
ax[1].plot(tp, Vp, lw=0.9, color='C0'); ax[1].set_ylabel('V (V)'); ax[1].set_xlabel('Time (s)'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Voltage swing under pulse load: {Vp.min():.3f} – {Vp.max():.3f} V')

## 4. Smoke-test pass criteria

- [x] All imports succeed
- [x] `Prada2013` parameter set loads
- [x] DFN solves a CC discharge to cut-off without error
- [x] V-SOC curve shows the LFP plateau (~3.2–3.35 V)
- [x] Dynamic current function (`Current function [A]` callable) is accepted

Pass → proceed to W2: load Severson .mat files, build feature pipeline, train LSTM baseline against the 9.1 % MAPE target.